In [ ]:
from pathlib import Path
import base64
import json
import os

import pandas as pd
import requests

import teehr
from teehr import RemoteReadWriteEvaluation
from teehr.utilities.apply_migrations import evolve_catalog_schema

from setup_utils import (
    DEV_LOCATION_ID_LIST,
    apply_polaris_token_to_spark,
    create_minio_spark_session,
    ensure_fresh_polaris_user_token,
)

#### Start the spark session

In [ ]:
import os
print("POLARIS_CLIENT_ID:", os.getenv("POLARIS_CLIENT_ID"))
print("POLARIS_CLIENT_SECRET exists:", bool(os.getenv("POLARIS_CLIENT_SECRET")))
print("POLARIS_REFRESH_TOKEN exists:", bool(os.getenv("POLARIS_REFRESH_TOKEN")))

In [ ]:
polaris_user_token = os.environ.get("POLARIS_USER_TOKEN")
polaris_refresh_token = os.getenv("POLARIS_REFRESH_TOKEN", "")
polaris_username = os.getenv("POLARIS_USERNAME", os.getenv("JUPYTERHUB_USER", "admin"))
polaris_password = os.getenv("POLARIS_PASSWORD", "")
polaris_client_id = os.getenv("POLARIS_CLIENT_ID", "jupyterhub")
polaris_client_secret = os.getenv("POLARIS_CLIENT_SECRET")

if not polaris_user_token and not polaris_refresh_token and not polaris_password:
    raise RuntimeError(
        "No usable Polaris credentials found. Set POLARIS_USER_TOKEN or POLARIS_REFRESH_TOKEN in the session."
    )

polaris_user_token, polaris_refresh_token, token_renewed = ensure_fresh_polaris_user_token(
    current_token=polaris_user_token,
    username=polaris_username,
    password=polaris_password,
    client_id=polaris_client_id,
    client_secret=polaris_client_secret,
    refresh_token=polaris_refresh_token,
    refresh_window_seconds=120,
)
os.environ["POLARIS_USER_TOKEN"] = polaris_user_token
if polaris_refresh_token:
    os.environ["POLARIS_REFRESH_TOKEN"] = polaris_refresh_token

payload = polaris_user_token.split(".")[1]
payload += "=" * (-len(payload) % 4)
claims = json.loads(base64.urlsafe_b64decode(payload.encode()))
roles = sorted(claims.get("realm_access", {}).get("roles", []))

print("Token status:", "renewed" if token_renewed else "reused")
print("Refresh token available:", bool(polaris_refresh_token))
print("Token user:", claims.get("preferred_username"))
print("Token roles:", ", ".join(roles))
if "iceberg-catalog-admin" not in roles:
    print("WARNING: token is missing iceberg-catalog-admin; Polaris may reject writes")

In [ ]:
import requests

realm = os.getenv("POLARIS_DEFAULT_REALM", "teehr")
remote_warehouse = os.getenv("REMOTE_WAREHOUSE_S3_PATH", "s3://warehouse/")
polaris_api_base = os.getenv("POLARIS_CATALOG_URI", "http://polaris:8181/api/catalog").rstrip("/")
catalog_config_url = f"{polaris_api_base}/v1/config"
mgmt_catalog_url = f"http://polaris:8181/api/management/v1/catalogs/{realm}"

# Refresh access token before probing Polaris or recreating Spark.
polaris_user_token, polaris_refresh_token, refreshed = ensure_fresh_polaris_user_token(
    current_token=polaris_user_token,
    username=polaris_username,
    password=polaris_password,
    client_id=polaris_client_id,
    client_secret=polaris_client_secret,
    refresh_token=polaris_refresh_token,
    refresh_window_seconds=120,
)
os.environ["POLARIS_USER_TOKEN"] = polaris_user_token
if polaris_refresh_token:
    os.environ["POLARIS_REFRESH_TOKEN"] = polaris_refresh_token
print("Cell 4 token status:", "renewed" if refreshed else "still fresh")

# Polaris REST config endpoint expects catalog identifier (realm), not S3 URI.
warehouse_for_config = remote_warehouse
if polaris_api_base.endswith("/api/catalog"):
    warehouse_for_config = realm

headers = {
    "Authorization": f"Bearer {polaris_user_token}",
    "X-Polaris-Realm": realm,
}

# Catalog config endpoint requires a warehouse query parameter.
catalog_resp = requests.get(
    catalog_config_url,
    headers=headers,
    params={"warehouse": warehouse_for_config},
    timeout=20,
 )
print("Catalog config API status:", catalog_resp.status_code)
print("Catalog config warehouse query:", warehouse_for_config)
if catalog_resp.status_code >= 400:
    print("Catalog config API body:", catalog_resp.text[:500])
catalog_resp.raise_for_status()

if "iceberg-catalog-admin" in roles:
    mgmt_resp = requests.get(mgmt_catalog_url, headers=headers, timeout=20)
    print("Management API status:", mgmt_resp.status_code)
    if mgmt_resp.status_code >= 400:
        print("Management API body:", mgmt_resp.text[:500])
    mgmt_resp.raise_for_status()
else:
    print(
        "Skipping Management API check: token does not include iceberg-catalog-admin. "
        "Catalog access is sufficient for non-admin Spark paths."
    )

print("Polaris access check passed.")

spark = create_minio_spark_session(
    polaris_token=polaris_user_token,
    force_recreate_session=True,
)


def refresh_polaris_token_for_spark(refresh_window_seconds: int = 120) -> bool:
    global polaris_user_token, polaris_refresh_token

    polaris_user_token, polaris_refresh_token, refreshed = ensure_fresh_polaris_user_token(
        current_token=polaris_user_token,
        username=polaris_username,
        password=polaris_password,
        client_id=polaris_client_id,
        client_secret=polaris_client_secret,
        refresh_token=polaris_refresh_token,
        refresh_window_seconds=refresh_window_seconds,
    )
    os.environ["POLARIS_USER_TOKEN"] = polaris_user_token
    if polaris_refresh_token:
        os.environ["POLARIS_REFRESH_TOKEN"] = polaris_refresh_token
    apply_polaris_token_to_spark(spark, polaris_user_token, catalog_name="iceberg", realm=realm)

    print("Spark token status:", "renewed" if refreshed else "still fresh")
    return refreshed


apply_polaris_token_to_spark(spark, polaris_user_token, catalog_name="iceberg", realm=realm)
print("Spark catalog namespace probe:")
spark.sql("SHOW NAMESPACES IN iceberg").show(truncate=False)

In [ ]:
os.environ["POLARIS_REFRESH_TOKEN"]

In [ ]:
migrations_dir = Path(teehr.__file__).parent / "migrations"

evolve_catalog_schema(
    spark=spark,
    migrations_dir_path=migrations_dir,
    target_catalog_name="iceberg",
    target_namespace_name="teehr"
)

#### Initialize a TEEHR Evaluation with read/write privileges

In [ ]:
ev = RemoteReadWriteEvaluation(spark=spark, enable_spark_proxy=True)

#### Now you can pull in a subset of data from the TEEHR-Cloud warehouse via the API
Note: If you encounter a timeout or auth error, run refresh_polaris_token_for_spark() and retry. Prefer setting POLARIS_REFRESH_TOKEN in the session so renewals do not depend on username/password.

Download domain variables, location data, and attribute data

In [ ]:
print("Loading configurations.")
ev.download.configurations(load=True)

print("Loading units.")
ev.download.units(load=True)

print("Loading variables.")
ev.download.variables(load=True)

print("Loading locations.")
ev.download.locations(
    ids=DEV_LOCATION_ID_LIST,
    load=True
)
print("Loading location croswalks.")
ev.download.location_crosswalks(
    primary_location_id=DEV_LOCATION_ID_LIST,
    load=True
)
print("Loading location attributes.")
ev.download.attributes(load=True)
ev.download.location_attributes(
    location_id=DEV_LOCATION_ID_LIST,
    load=True
)
print("Loading domain variables, locations, crosswalks, and attributes complete!")

Download forecast timeseries and obs

In [ ]:
ev.download.primary_timeseries(
    primary_location_id=DEV_LOCATION_ID_LIST,
    configuration_name="usgs_observations",
    variable_name="streamflow_hourly_inst",
    start_date="2026-05-01",
    end_date="2026-05-12",
    load=True,
)

ev.download.secondary_timeseries(
    primary_location_id=DEV_LOCATION_ID_LIST,
    configuration_name="nwm30_medium_range",
    variable_name="streamflow_hourly_inst",
    reference_time="2026-05-01/2026-05-02",
    load=True,
)

ev.download.secondary_timeseries(
    primary_location_id=DEV_LOCATION_ID_LIST,
    configuration_name="nwm30_short_range_hawaii",
    variable_name="streamflow_15min_inst",
    reference_time="2026-05-01/2026-05-10",
    load=True,
)

ev.download.secondary_timeseries(
    primary_location_id=DEV_LOCATION_ID_LIST,
    configuration_name="nwm30_short_range_alaska",
    variable_name="streamflow_hourly_inst",
    reference_time="2026-05-01/2026-05-10",
    load=True,
)

ev.download.secondary_timeseries(
    primary_location_id=DEV_LOCATION_ID_LIST,
    configuration_name="nwm30_short_range_puertorico",
    variable_name="streamflow_hourly_inst",
    reference_time="2026-05-01/2026-05-10",
    load=True,
)
print("Loading forecast timeseries complete!")

Download retrospective simulation timeseries and obs

In [ ]:
ev.download.primary_timeseries(
    primary_location_id=DEV_LOCATION_ID_LIST,
    configuration_name="usgs_observations",
    variable_name="streamflow_hourly_inst",
    start_date="2020-05-01",
    end_date="2020-05-31",
    load=True,
)

ev.download.secondary_timeseries(
    primary_location_id=DEV_LOCATION_ID_LIST,
    configuration_name="nwm30_retrospective",
    variable_name="streamflow_hourly_inst",
    start_date="2020-05-01",
    end_date="2020-05-31",
    load=True,
)
print("Loading retrospective timeseries complete!")

Optional: To run the NWM forcing workflows you need to add the basin pixel weights table manually at this time
- You need to copy the `examples/developer/local_data` directory into your `data` dir

In [ ]:
# Load the pre-created weights table from local data
df = pd.read_parquet("local_data/dev_grid_pixel_coverage_weights.parquet")
ev.table("grid_pixel_coverage_weights").load_dataframe(
    df=df,
    write_mode="create_or_replace"
)
# Download the corresponding USGS basin locations
ev.download.locations(
    ids=df.location_id.unique().tolist(),
    load=True
)
print("Loading forcing pixel weights table complete")

Now download HUC6 and USGS basins for the data management dashboard

In [ ]:
ev.download.locations(prefix="huc6", load=True)

In [ ]:
basin_ids = [gage_id.replace("usgs-", "usgsbasin-") for gage_id in DEV_LOCATION_ID_LIST]
ev.download.locations(ids=basin_ids, load=True)

To enable the data management dashboards run the following prefect workflows:
- `update-configurations-summary-table`
- `update-locations-summary-table`
- `update-completeness-summary-table` (requires huc6 basins)

For the retrospective dashboard run the following notebooks:
- `02_create_joined_timeseries.ipynb`
- `03_generate_basic_metrics notebooks.ipynb`

For the forecast dashboard run the following prefect workflows:
- `update-joined-forecast-table`
- `update-forecast-metrics-table`

In [ ]:
ev.configurations.to_sdf().show()

In [ ]:
spark.stop()